# TV-00a C# — RoPE from scratch : coder la position par rotation

Le notebook d’architecture (3.4 / GenAI-Texte 03) implémente l’attention canonique avec un **encodage positionnel additif** — on ajoute un vecteur de position à l’embedding. RoPE (*Rotary Position Embedding*, Su et al. 2021) fait un choix différent : il **fait tourner** les paires de dimensions de `q` et `k` d’un angle proportionnel à la position. C’est le schéma de position retenu par LLaMA-2/3, Mistral, Qwen et Gemma.

**Pourquoi ce notebook existe** : ouvrir la boîte. `AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.1")` cache un `rope_theta=10000` et des paires de dimensions qui tournent — ici on le code avec TorchSharp et les calculs scalaires C# et on **mesure** ce que ce choix apporte.

**Les quatre résultats mesurés** (chacun est vérifiable en relançant les cellules) :

1. `R(m)` est construite par blocs de rotation orthogonaux : elle préserve les normes.
2. La distance `d(m, n)` entre `q` en position `m` et `k` en position `n` ne dépend **que de l’écart `m − n`** — et la variante additive héritée de la source sert de témoin numérique.
3. Le **score d’attention** `(R_m q) · (R_n k)` ne voit que la position **relative** : translater toute la séquence ne change aucune logit.
4. Sur une tâche d’inversion de séquence entraînée pour de vrai, les deux PE apprennent, mais leur comportement **hors de la longueur d’entraînement** les sépare.

Tout est réimplémenté avec TorchSharp et les calculs scalaires C# : aucun appel à `transformers`, `xformers` ni `flash-attn`.

[Version Python](TV-00a-RoPE-from-scratch.ipynb) | [TV-00b C# · Variantes d’attention →](TV-00b-Attention-Variants-from-scratch-CSharp.ipynb)

## 0. Mise en place

Échelle pédagogique volontairement minuscule (dimension 16 pour la géométrie, 32 pour la tâche, CPU) : RoPE est une propriété **algébrique**, elle se démontre à n'importe quelle échelle.

In [1]:
#r "nuget:TorchSharp-cpu, 0.107.0"
// Un seul thread libtorch rend le petit banc comparable et limite la variabilité des réductions.
using System;
using System.Linq;
using System.Collections.Generic;
using TorchSharp;
using static TorchSharp.torch;
using static TorchSharp.torch.nn;
using Tensor = TorchSharp.torch.Tensor;

torch.set_num_threads(1);
torch.manual_seed(0);
Console.WriteLine($"TorchSharp {typeof(torch).Assembly.GetName().Version} ; libtorch {torch.__version__} ; CPU, threads {torch.get_num_threads()}");


Installed Packages TorchSharp-cpu, 0.107.0

TorchSharp 0.107.0.0 ; libtorch 2.10.0.0 ; CPU, threads 1


## 1. La géométrie : une rotation par paire de dimensions

RoPE découpe le vecteur de dimension `d` en `d/2` **paires** `(x_{2i}, x_{2i+1})` et fait tourner la paire `i` d'un angle `m · θ_i` quand le token est en position `m` :

$$R(m) = \begin{pmatrix} \cos m\theta_0 & -\sin m\theta_0 & & & \\ \sin m\theta_0 & \cos m\theta_0 & & & \\ & & \ddots & & \\ & & & \cos m\theta_{d/2-1} & -\sin m\theta_{d/2-1} \\ & & & \sin m\theta_{d/2-1} & \cos m\theta_{d/2-1} \end{pmatrix}, \qquad \theta_i = \text{base}^{-2i/d}$$

`R(m)` est la matrice bloc-diagonale des rotations planes. La **fréquence** `θ_i` décroît avec `i` : la première paire tourne vite (elle encode la localité fine), la dernière tourne presque immobile (elle encode la position grossière).

In [2]:
// Chaque paire reçoit une fréquence sur une progression géométrique.
double[] Theta(int dim, double b = 10000)
{
    if (dim % 2 != 0)
        throw new ArgumentException("dimension paire requise");
    return Enumerable.Range(0, dim / 2).Select(i => Math.Pow(b, -2.0 * i / dim)).ToArray();
}

double[, ] Matrix(int m, int dim)
{
    var theta = Theta(dim);
    var R = new double[dim, dim];
    // La matrice est bloc-diagonale : chaque bloc tourne seulement deux coordonnées.
    for (int i = 0; i < theta.Length; i++)
    {
        var a = m * theta[i];
        R[2 * i, 2 * i] = Math.Cos(a);
        R[2 * i, 2 * i + 1] = -Math.Sin(a);
        R[2 * i + 1, 2 * i] = Math.Sin(a);
        R[2 * i + 1, 2 * i + 1] = Math.Cos(a);
    }

    return R;
}

int DIM_DEMO = 16;
var theta = Theta(DIM_DEMO);
Console.WriteLine($"dimension {DIM_DEMO}, paires {theta.Length}");
Console.WriteLine("theta : " + string.Join(", ", theta.Select(x => x.ToString("G6"))));
Console.WriteLine("longueurs d'onde : " + string.Join(", ", theta.Select(x => (2 * Math.PI / x).ToString("F1"))));

dimension 16, paires 8


theta : 1, 0.316228, 0.1, 0.0316228, 0.01, 0.00316228, 0.001, 0.000316228


longueurs d'onde : 6.3, 19.9, 62.8, 198.7, 628.3, 1986.9, 6283.2, 19869.2


### Lire les fréquences : une échelle géométrique de localité

La sortie imprime huit fréquences et huit longueurs d'onde. Elles forment une **échelle géométrique** : chaque paire de dimensions tourne `10000^(2/16) = 3,16` fois moins vite que la précédente. Les extrêmes :

- **paire 0** : longueur d'onde **6,3** positions. Elle accomplit un tour complet en six tokens — c'est le code de **localité fine** : deux tokens éloignés de 3 positions ont des angles très différents sur cette paire.
- **paire 7** : longueur d'onde **20 000** positions. Sur un contexte de 2 000 tokens elle n'a tourné que de `2000/20000 × 360° = 36°` — quasi immobile, elle porte la **position grossière**.

L'écart entre les deux extrêmes est un facteur `20 000 / 6,3 ≈ 3 200` : RoPE dédie chaque paire de dimensions à une **échelle de distance** différente, du « voisin immédiat » au « rang approximatif dans le document ». C'est la même astuce que les harmoniques du PE additif sinusoïdal — mais au lieu d'*additionner* ces sinusoïdes au vecteur, RoPE les utilise comme **vitesses de rotation**.

### Exercice 1 — la base élargie de LLaMA-3

LLaMA-3 utilise `rope_theta = 500000` (contre `10000` historiquement) pour étendre le contexte. Écrire `ThetaLongContext(dim)` qui retourne les fréquences pour `base = 5e5`, puis **mesurer** le rapport entre la plus grande et la plus petite longueur d'onde, et comparer à `base = 1e4` : quelles paires de dimensions cela change-t-il, et pourquoi cela aide-t-il les longs contextes ?

**Indice :** seules les paires à basse fréquence (grands `i`) sont sensiblement modifiées ; exprimer le rapport des longueurs d'onde extrêmes `λ_max/λ_min` pour les deux bases.
- Étapes : (1) réutiliser `Theta` avec la nouvelle base ; (2) calculer les deux rapports ; (3) conclure sur les dimensions concernées.

In [3]:
double[] ThetaLongContext(int dim, double baseValue = 500000)
{ // TODO étudiant : fréquences et comparaison des longueurs d’onde
    return null;
}

Console.WriteLine("Exercice 1 à compléter");


Exercice 1 à compléter


## 2. L'invariance relative — le coeur du notebook

La grandeur qui intéresse l'attention n'est pas la position absolue : c'est la **distance** entre la requête en position `m` et la clé en position `n` :

$$d^2(m, n) = \|R_m\, q - R_n\, k\|^2 = \|q\|^2 + \|k\|^2 - 2\, q^\top R_m^\top R_n\, k$$

Or `R_m^T R_n = R_{n-m}` (groupe des rotations). Donc **tout le contenu positionnel de la distance se réduit à `R_{n−m}`** : si RoPE tient sa promesse, `d(m, n)` ne doit dépendre que de l'écart `n − m`. C'est falsifiable : mesurons.

In [4]:
// La rotation conserve les paires adjacentes et travaille en double pour contrôler les écarts numériques.
Tensor RopeVector(Tensor x, double position, int dim)
{
    var theta = (-Math.Log(10000.0) * 2 * torch.arange(dim / 2, dtype: ScalarType.Float64) / dim).exp();
    var pairs = x.to_type(ScalarType.Float64).reshape(dim / 2, 2);
    var angles = theta * position;
    var even = pairs.select(-1, 0);
    var odd = pairs.select(-1, 1);
    return torch.stack(new[] { even * angles.cos() - odd * angles.sin(), even * angles.sin() + odd * angles.cos() }, -1).reshape(dim);
}

// La même graine donne le même couple q, k pour toutes les positions testées.
var g = new torch.Generator(7, torch.CPU);
var q = torch.randn(new long[] { DIM_DEMO }, generator: g, dtype: ScalarType.Float64);
var k = torch.randn(new long[] { DIM_DEMO }, generator: g, dtype: ScalarType.Float64);
double DistRope(int m, int n)
{
    using var scope = torch.NewDisposeScope();
    var diff = RopeVector(q, m, DIM_DEMO) - RopeVector(k, n, DIM_DEMO);
    return diff.square().sum().item<double>();
}

var distances = new[]
{
    DistRope(3, 7),
    DistRope(10, 14),
    DistRope(100, 104)
};
Console.WriteLine("Distances pour écart 4 : " + string.Join(", ", distances.Select(x => x.ToString("G17"))));
var delta = distances.Max() - distances.Min();
Console.WriteLine($"Écart maximum : {delta:G17} ; tolérance float64 = 1e-12");
if (delta > 1e-12)
    throw new Exception("invariance relative non vérifiée");
Console.WriteLine($"ordre inversé : {DistRope(7, 3):G17} ; écart 7 : {DistRope(3, 10):G17}");

Distances pour écart 4 : 34.746126188450376, 34.746126188450368, 34.746126188450347


Écart maximum : 2.8421709430404007E-14 ; tolérance float64 = 1e-12


ordre inversé : 25.115562087639461 ; écart 7 : 41.310707195064332


Comparez les trois distances : un même décalage relatif conserve la distance à la tolérance affichée. Inverser l’ordre change le décalage signé et peut changer la distance.

### Exercice 2 — le RoPE partiel (partial rotary)

GPT-NeoX et Phi « tournent » seulement une **fraction** des dimensions (typiquement 25 %) et laissent le reste sans position. Écrire `PartialRope(x, positions, dim, fraction: 0.25)` qui n'applique la rotation qu'aux premières paires, puis **mesurer** ce que devient l'invariance de la section 2 : que reste-t-il du signal de position dans chaque bloc ?

**Indice :** décomposer `d^2` en partie tournée (invariante) et partie libre, à exprimer en fonction de ses composantes de `q − k` ; comparer `partial d^2(3,7)` et `partial d^2(100,104)`.
- Étapes : (1) construire la rotation sur `int(fraction * dim/2)` paires ; (2) recalculer les trois distances de la section 2 ; (3) identifier ce que la partie non tournée conserve comme signal de position.

In [5]:
Tensor PartialRope(Tensor x, Tensor positions, int dim, double fraction = 0.25)
{ // TODO étudiant : tourner seulement la fraction demandée
    return null;
}

Console.WriteLine("Exercice 2 à compléter");


Exercice 2 à compléter


### Le témoin sinusoïdal additif du notebook source

Un encodage sinusoïdal additif ajoute un vecteur de position `p_m`. Ici, `AdditivePE` reproduit la variante du carnet Python. Dans la paire `j`, la cellule prend `i = 2j`, puis applique le facteur `2i/d`. La fréquence effective vaut donc $10000^{-4j/d}$, tandis que le PE sinusoïdal original utilise $10000^{-2j/d}$. La paire contient le sinus et le cosinus de la position multipliée par la fréquence.

La distance additive sur des vecteurs de contenu distincts dépend du terme `p_m − p_n`. Comparons maintenant plusieurs paires ayant le même décalage.


In [6]:
// L’encodage additif se calcule avec la position absolue avant la somme au vecteur.
Tensor AdditivePE(int pos, int dim)
{
    var i = torch.arange(0, dim, 2, dtype: ScalarType.Float64);
    var a = pos * (-Math.Log(10000.0) * 2 * i / dim).exp();
    return torch.stack(new[] { a.sin(), a.cos() }, -1).reshape(dim);
}

double DistAdd(int m, int n)
{
    using var scope = torch.NewDisposeScope();
    var d = (q + AdditivePE(m, DIM_DEMO)) - (k + AdditivePE(n, DIM_DEMO));
    return d.square().sum().item<double>();
}

var a3 = DistAdd(3, 7);
var a10 = DistAdd(10, 14);
var a100 = DistAdd(100, 104);
Console.WriteLine($"PE additif : {a3:G17}, {a10:G17}, {a100:G17} ; écart relatif {Math.Abs(a100 - a3) / a3:G6}");

PE additif : 25.989631900257791, 27.083013312960773, 30.381994291814834 ; écart relatif 0.169004


### Lecture du témoin additif

Pour le même écart de quatre positions, la variante du carnet source donne `25,99`, `27,08` et `30,38`, soit un écart relatif de **16,9 %** entre les deux mesures extrêmes. Les distances RoPE de la section précédente coïncident à la tolérance testée.

Pour des vecteurs de contenu fixés, la distance additive contient un produit croisé avec `p_m − p_n`. Les composantes de cette différence changent avec le point de départ même à décalage fixé. Avec RoPE, la composition des rotations dépend du décalage seul. Cette propriété algébrique et la généralisation après entraînement sont évaluées séparément.


## 3. Dans le score d'attention : la position absolue disparaît

L'attention calcule un produit scalaire. Avec RoPE appliqué à `q` et `k` :

$$s(m, n) = (R_m\, q)^\top (R_n\, k) = q^\top \underbrace{R_m^\top R_n}_{=\,R_{n-m}}\, k$$

Le score ne dépend que de la rotation **relative** `R_{n−m}`. Conséquence testable : décaler toute la séquence de `Δ` positions (même contenu, positions 43 et 47 au lieu de 3 et 7) doit laisser le score **inchangé** — et le score additif, lui, doit bouger.

In [7]:
// Le produit scalaire contrôle la propriété utile pour les logits de l’attention.
double ScoreRope(int m, int n)
{
    using var scope = torch.NewDisposeScope();
    return (RopeVector(q, m, DIM_DEMO) * RopeVector(k, n, DIM_DEMO)).sum().item<double>();
}

double ScoreAdd(int m, int n)
{
    using var scope = torch.NewDisposeScope();
    return ((q + AdditivePE(m, DIM_DEMO)) * (k + AdditivePE(n, DIM_DEMO))).sum().item<double>();
}

// Comparer la même paire de vecteurs avec et sans translation commune des positions.
var sr1 = ScoreRope(3, 7);
var sr2 = ScoreRope(43, 47);
var sa1 = ScoreAdd(3, 7);
var sa2 = ScoreAdd(43, 47);
Console.WriteLine($"RoPE : {sr1:G17}, {sr2:G17}, écart {Math.Abs(sr1 - sr2):G17}");
Console.WriteLine($"Additif : {sa1:G17}, {sa2:G17}, écart {Math.Abs(sa1 - sa2):G17}");
if (Math.Abs(sr1 - sr2) > 1e-12 || Math.Abs(sa1 - sa2) < 1e-8)
    throw new Exception("contraste des logits non vérifié");

RoPE : 6.0913595524747635, 6.0913595524747635, écart 0


Additif : 21.042455600303562, 13.999114522131563, écart 7.0433410781719985


### Lecture : translater la séquence ne change aucune logit RoPE

La cellule précédente compare deux scores séparés par une translation commune et vérifie une tolérance explicite en double précision. Le produit scalaire RoPE ne dépend que du décalage relatif, conformément à l’algèbre `qᵀ R_{n−m} k`. Les scores additifs, calculés pour les mêmes vecteurs, diffèrent ; leur écart est affiché et contrôlé. Ce résultat vaut pour les vecteurs de ce test, tandis que l’invariance RoPE découle de la composition des rotations.

### Exercice 3 — l'interpolation linéaire de position

Pour étendre le contexte d'un modèle déjà entraîné (8k → 32k), la technique historique divise les positions par un facteur `s` avant rotation (« position interpolation »). Écrire `DistRopeScaled(m, n, scale)` qui calcule `d^2` avec les positions `m/s` et `n/s`, puis **mesurer** ce que devient la distance aux grands écarts : que perd-on à interpoler ?

**Indice :** `R_{n−m}` devient `R_{(n−m)/s}` — l'invariance relative survit-elle ? Auxquels des résultats de la section 2 cette transformation touche-t-elle, et auxquels elle ne touche pas ?
- Étapes : (1) réutiliser `RopeVector` avec les positions divisées par `scale` ; (2) comparer `d^2(100, 104)` pour `s = 1` et `s = 4` ; (3) conclure sur la résolution positionnelle perdue.

In [8]:
double? DistRopeScaled(int m, int n, double scale)
{ // TODO étudiant : interpoler les positions avant rotation
    return null;
}

Console.WriteLine("Exercice 3 à compléter");


Exercice 3 à compléter


## 4. Tâche d'inversion : PE additif vs RoPE, entraînés pour de vrai

**La tâche** : recevoir une séquence de 12 tokens (vocabulaire 10) et produire la séquence **inversée**. C'est un test de position pur — le token en sortie `t` doit aller chercher l'information en entrée `S−1−t`, un routage purement positionnel.

**Pourquoi l'attention n'est PAS causale ici** : avec un masque causal, la sortie au rang `t` ne voit que les rangs `≤ t` — inverser exige de lire le rang `S−1−t` qui est **après** `t` pour toute la première moitié. La tâche serait structurellement impossible ; ce notebook étudie l'encodage de position, pas le langage causal, donc l'attention est bidirectionnelle (comme un encodeur BERT).

**Le protocole** : même architecture (embedding, une tête d'attention `W_q W_k W_v`, tête linéaire), même budget (Adam, même nombre de pas), seule change l'injection de position — ajout du vecteur sinusoïdal du carnet source sur `q` et `k` (fréquence $10000^{-4j/d}$, voir § 2), ou rotation RoPE de ces deux tenseurs. Les résultats comparent ces deux implémentations sur cette tâche. Trois graines par configuration (la mesure d'une seule graine ne vaudrait pas conclusion). Puis on évalue **aux deux longueurs** : 12 (vue à l'entraînement) et 16 (jamais vue).

In [9]:
// Les logits gardent les projections Q/K distinctes. La rotation n’ajoute aucun paramètre à entraîner.
public class Tiny : Module<Tensor, Tensor>
{
    readonly int dim;
    readonly string pos;
    readonly Module<Tensor, Tensor> emb, wq, wk, wv, head;
    public Tiny(int d, int v, string p) : base("Tiny")
    {
        dim = d;
        pos = p;
        emb = Embedding(v, d);
        wq = torch.nn.Linear(inputSize: d, outputSize: d, hasBias: false);
        wk = torch.nn.Linear(inputSize: d, outputSize: d, hasBias: false);
        wv = torch.nn.Linear(inputSize: d, outputSize: d, hasBias: false);
        head = torch.nn.Linear(inputSize: d, outputSize: v);
        RegisterComponents();
    }

    // Calculer les phases en double puis revenir au dtype des projections.
    Tensor Rope(Tensor x, long seq)
    {
        var theta = (-Math.Log(10000.0) * 2 * arange(dim / 2, dtype: ScalarType.Float64) / dim).exp();
        var pairs = x.to_type(ScalarType.Float64).reshape(x.shape[0], seq, dim / 2, 2);
        var angles = arange(seq, dtype: ScalarType.Float64).unsqueeze(1) * theta.unsqueeze(0);
        var cos = angles.cos();
        var sin = angles.sin();
        var even = pairs.select(-1, 0);
        var odd = pairs.select(-1, 1);
        return stack(new[] { even * cos - odd * sin, even * sin + odd * cos }, -1).reshape(x.shape).to_type(x.dtype);
    }

    Tensor PE(long seq)
    {
        var i = arange(0, dim, 2, dtype: ScalarType.Float64);
        var angle = arange(seq, dtype: ScalarType.Float64).unsqueeze(1) * (-Math.Log(10000) * 2 * i / dim).exp().unsqueeze(0);
        return stack(new[] { angle.sin(), angle.cos() }, -1).reshape(seq, dim).to_type(ScalarType.Float32);
    }

    public override Tensor forward(Tensor t)
    {
        var x = emb.forward(t);
        var q = wq.forward(x);
        var k = wk.forward(x);
        var v = wv.forward(x);
        long seq = t.shape[1];
        if (pos == "add")
        {
            var pe = PE(seq);
            q = q + pe;
            k = k + pe;
        }
        else
        {
            q = Rope(q, seq);
            k = Rope(k, seq);
        }

        var scores = matmul(q, k.transpose(-1, -2)) / Math.Sqrt(dim);
        return head.forward(matmul(scores.softmax(-1), v));
    }
}

Console.WriteLine("Architecture : dim 32, vocabulaire 10, séquences 12 / 16");
using (var test = new Tiny(32, 10, "rope"))
{
    var count = test.parameters().Sum(p => p.numel());
    Console.WriteLine($"Paramètres entraînables : {count}");
    if (count != 3722)
        throw new Exception("compte de paramètres inattendu");
}

Architecture : dim 32, vocabulaire 10, séquences 12 / 16


Paramètres entraînables : 3722


Le compte de paramètres est vérifié : embedding, trois projections sans biais et tête de sortie avec biais.

In [10]:
(Tensor, Tensor) Batch(int batch, int seq, torch.Generator generator)
{
    var x = torch.randint(1, 10, new long[] { batch, seq }, generator: generator);
    return (x, x.flip(new long[] { 1 }));
}

Tiny Train(string pos, int seed)
{
    torch.manual_seed(seed);
    var model = new Tiny(32, 10, pos);
    using var opt = torch.optim.Adam(model.parameters(), lr: 0.003);
    // La graine de génération des lots est distincte de l’initialisation du modèle.
    var gen = new torch.Generator((ulong)(seed + 1), torch.CPU);
    for (int step = 1; step <= 400; step++)
    {
        using var scope = torch.NewDisposeScope();
        var(x, y) = Batch(64, 12, gen);
        var loss = torch.nn.functional.cross_entropy(model.forward(x).reshape(-1, 10), y.reshape(-1));
        opt.zero_grad();
        loss.backward();
        opt.step();
        if (step % 100 == 0)
            Console.WriteLine($"{pos} graine {seed} pas {step} perte {loss.item<float>():F4}");
    }

    return model;
}

// Le test fixe ses exemples sans en faire un lot de mise à jour.
double Accuracy(Tiny model, int seq)
{
    using var scope = torch.NewDisposeScope();
    using var ng = torch.no_grad();
    var(x, y) = Batch(256, seq, new torch.Generator(999, torch.CPU));
    return model.forward(x).argmax(-1).eq(y).to_type(ScalarType.Float32).mean().item<float>();
}

// Les générateurs de données sont indépendants des initialisations des modèles.
using (var checkScope = torch.NewDisposeScope())
{
    var(x1, y1) = Batch(64, 12, new torch.Generator(1, torch.CPU));
    var(x2, y2) = Batch(64, 12, new torch.Generator(1, torch.CPU));
    if ((x1 - x2).abs().max().item<long>() != 0 || (y1 - y2).abs().max().item<long>() != 0)
        throw new Exception("lots non reproductibles");
    Console.WriteLine("Identité des lots issus de la même graine : vérifiée");
}

foreach (var pos in new[]
{
    "add",
    "rope"
}

)
{
    var seen = new List<double>();
    var unseen = new List<double>();
    foreach (var seed in new[]
    {
        0,
        1,
        2
    }

    )
    {
        using var model = Train(pos, seed);
        var a = Accuracy(model, 12);
        var b = Accuracy(model, 16);
        seen.Add(a);
        unseen.Add(b);
        Console.WriteLine($"{pos}, graine {seed} : vue {a:F4}, non vue {b:F4}");
    }

    Console.WriteLine($"MOYENNE {pos} : vue {seen.Average():F4}, non vue {unseen.Average():F4}");
}

Identité des lots issus de la même graine : vérifiée


add graine 0 pas 100 perte 1.9875


add graine 0 pas 200 perte 1.9587


add graine 0 pas 300 perte 1.9414


add graine 0 pas 400 perte 1.9457


add, graine 0 : vue 0.2676, non vue 0.2402


... (23 outputs left out)

rope graine 2 pas 300 perte 1.6469


rope graine 2 pas 400 perte 1.6547


rope, graine 2 : vue 0.3854, non vue 0.2349


MOYENNE rope : vue 0.3855, non vue 0.2205


### Lecture : ce que la tâche mesure — et la surprise qu'elle réserve

**Ce que le banc établit solidement.** RoPE apprend mieux cette tâche, à budget identique : perte finale ~1,66 contre ~1,94, et exactitude token **0,386 contre 0,266** — douze points d'écart, les trois graines affichées permettent d’apprécier la dispersion à cette longueur.

**Ce que le banc ne résout pas.** Personne ne maîtrise la tâche : le hasard vaut `1/9 ≈ 0,111` (vocabulaire effectif de 9 symboles), les deux modèles restent entre 2× et 3,5× au-dessus. C'est un banc délibérément minuscule (3 722 paramètres, 400 pas) — l’invariance algébrique des rotations, elle, est établie aux sections 2 et 3.

**La surprise mesurée.** À la longueur 16, jamais vue, l'ordre **s'inverse** : additif **0,241**, RoPE **0,221** (moyenne sur les trois graines ; les résultats individuels restent visibles). L'idée reçue « RoPE généralise mieux aux longueurs non vues » **ne se reproduit pas sur ce banc**. Une explication plausible : les scores RoPE ne consomment que des rotations relatives `R_{n−m}` ; la tâche d'inversion exige des offsets qui, à `S = 16`, sortent de la plage entraînée (jusqu'à ±15 contre ±11) — des angles jamais optimisés pendant l'apprentissage. Le PE additif, lui, injecte le signal absolu sinusoïdal de cette implémentation. Les positions 12 à 15 se situent au-delà de la plage connue ; l’expérience mesure leur effet pour ce banc.

La leçon n'est pas « le PE additif est meilleur » — à longueur vue, il perd nettement. Elle est que la **propriété brute** d'invariance relative ne suffit pas à généraliser : les modèles industriels qui étendent le contexte avec RoPE l'accompagnent de techniques dédiées — **interpolation de position** (exercice 3) et **base élargie** (exercice 1). Une affirmation reçue se mesure sur le banc qui la concerne : ici, elle échoue.

## 5. Bilan des exercices

Comparer la base, la rotation partielle et l’interpolation. Ces réglages touchent des propriétés différentes de l’encodage positionnel.

## Résumé

Les rotations préservent les produits scalaires sous translation commune. Le banc entraîne réellement les deux variantes et mesure les séquences vues et non vues ; utilisez le tableau précédent pour comparer ce replay.

## 6. Où ce notebook s'arrête, et ce qui suit

Ce notebook établit la **mécanique positionnelle** de RoPE. Les notebooks suivants de la série complètent le tableau des variantes modernes :

- **[TV-00b — Variantes d’attention C#](TV-00b-Attention-Variants-from-scratch-CSharp.ipynb)** (bloc A.1 de #16058) : MHA vs MQA vs GQA vs SWA — la réduction du KV-cache et la fenêtre glissante de Mistral, mesurées.
- **TV-00c — Mixture of Experts** (bloc A.3) : routage top-k, capacité, perte d'équilibrage.
- **Bloc B — SOTA** : charger un vrai modèle GQA+RoPE+SWA via `transformers` et confronter ses mesures aux implémentations from-scratch de cette série.

Voir #16058 pour l'acceptation complète.

> **Provenance de la série.** Ce carnet correspond au bloc A.2 de l’epic « Variantes Transformer modernes » ([#16058](https://github.com/jsboige/CoursIA/issues/16058)).